# 11 — Exploratory Data Analysis for Clustering (Objective 1)

**Objective 1:** *"To find which operational and infrastructure conditions explain
the differences in shipment weight and reported problems across the 25,000 warehouses."* The method
it commits to is *"clustering algorithms such as K-Means and hierarchical clustering to form
performance segments, followed by cluster profiling to identify the drivers within each segment."*

K-Means and hierarchical clustering group warehouses by **distance** — how far apart two warehouses
are across the columns they are given. So before any model is built, this notebook examines the data
from that angle and answers five questions:

| § | Question | Analysis |
|---|---|---|
| 1 | Which columns measure *performance*, and which describe *conditions* that might explain it? | role of each column |
| 2 | On what scales and in what shapes are the performance measures recorded? | univariate |
| 3 | What do the conditions look like, and how many columns would encoding them create? | univariate |
| 4 | Do any performance measures duplicate each other? | bivariate |
| 5 | Which conditions are related to performance at all — and to each other? | bivariate, with ANOVA, Kruskal–Wallis and chi-square |
| 6 | Is there any group structure for clustering to find, and in which set of columns? | multivariate — PCA and the Hopkins statistic |
| 7 | What does this mean for how the clustering is designed? | decisions opened |

**Input:** `data/preprocessed/warehouse_preprocessed.csv`
**Output:** none. This notebook reports only; it transforms and saves nothing.

This notebook explores the dataset to understand which columns and relationships are relevant for clustering, and identifies the four input columns and clustering scope that the transformation and modelling steps will use.

## 0. Setup


In [ ]:
import sys, pathlib

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *

set_style()

df = load_preprocessed()

# A labelled copy used only for plot colours: the 908 unrated warehouses (identified in
# the preliminary analysis) are shown separately throughout, because they may dominate any structure found.
plot_df = df.assign(warehouse=df["is_unrated_warehouse"].map({0: "rated", 1: "unrated (908)"}))

print(f"loaded {df.shape[0]:,} rows x {df.shape[1]} columns from {PREPROCESSED_FILE.name}")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors

---
## 1. Performance measures and conditions

The objective describes the problem in two parts. What *performance* looks like:

- *"differences in **shipment weight** and **reported problems**"* — Objective 1
- *"repeated **storage, transport** and staffing problems, and **breakdowns** halt work"* — Executive summary
- dashboards that *"report **refills, storage issues, breakdowns and shipment volumes**"* — Preliminary findings

And what might *explain* it: *"operational and **infrastructure conditions**"*, which cluster profiling
is to use *"to identify the drivers within each segment"*.

Read against the data dictionary, that suggests a role for every column. **This is a proposal drawn
from the wording of the objective, not a finding** — §5 and §6 test whether the data supports it.

| Role | Columns | Why |
|---|---|---|
| **Performance measures** | `product_wg_ton`, `num_refill_req_l3m`, `storage_issue_reported_l3m`, `transport_issue_l1y`, `wh_breakdown_l3m` | shipment volume, refills and the three reported problems used to define warehouse performance |
| **Conditions — numeric** | `workers_num`, `wh_est_year`, `dist_from_hub`, `Competitor_in_mkt`, `retail_shop_num`, `distributor_num`, `govt_check_l3m` | staffing, age, logistics, market and oversight |
| **Conditions — 0/1** | `electric_supply`, `temp_reg_mach`, `flood_proof`, `flood_impacted`, `is_unrated_warehouse` | infrastructure, site risk, certification status |
| **Conditions — categorical** | `Location_type`, `WH_capacity_size`, `zone`, `WH_regional_zone`, `wh_owner_type`, `approved_wh_govt_certificate` | location, size, ownership, certification |
| **Row key** | `Ware_house_ID` | identifies warehouses; never an input |
| **Recording flag** | `wh_est_year_missing` | describes how the data was recorded, not the warehouse; used only to restrict year-based analysis to recorded years (as established in preprocessing) |

`govt_check_l3m` is treated as a condition rather than a performance measure: it counts inspections by
an outside body, which is outside the tracked company measures.

In [ ]:
key = "Ware_house_ID"
performance = ["product_wg_ton", "num_refill_req_l3m", "storage_issue_reported_l3m",
               "transport_issue_l1y", "wh_breakdown_l3m"]
conditions_numeric = ["workers_num", "wh_est_year", "dist_from_hub", "Competitor_in_mkt",
                      "retail_shop_num", "distributor_num", "govt_check_l3m"]
conditions_binary = ["electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted",
                     "is_unrated_warehouse"]
conditions_categorical = ["Location_type", "WH_capacity_size", "zone", "WH_regional_zone",
                          "wh_owner_type", "approved_wh_govt_certificate"]
recording_flags = ["wh_est_year_missing"]

assigned = [key] + performance + conditions_numeric + conditions_binary + conditions_categorical + recording_flags
assert sorted(assigned) == sorted(df.columns), set(assigned) ^ set(df.columns)
assert len(assigned) == len(set(assigned)), "a column was assigned twice"

for name, cols in [("row key", [key]), ("performance measures", performance),
                   ("conditions - numeric", conditions_numeric), ("conditions - 0/1", conditions_binary),
                   ("conditions - categorical", conditions_categorical), ("recording flag", recording_flags)]:
    print(f"{name:<26} {len(cols):>2}")
print(f"{'total':<26} {len(assigned):>2}  of {df.shape[1]} columns")

> **Interpretation.**
>
> - All 25 columns are assigned to exactly one role with nothing left over: 5 performance
>   measures, 18 conditions (7 numeric, 5 on/off, 6 categorical), the row key and the recording flag. The two
>   assertions guarantee that no column is silently left out of, or counted twice in, the analysis below.


---
## 2. Univariate — the performance measures

For a distance-based method, two properties of each column matter before anything else:

- **Scale.** Distance adds up differences across columns. A column recorded in thousands contributes
  differences in the thousands; a 0/1 column contributes at most 1. Unless something is done about it,
  the large-scale column decides the clusters on its own.
- **Shape.** Spikes, gaps and long tails pull cluster centres toward them.

The preliminary analysis examined shape for every column. Here the question is narrower: *if these columns were given
to K-Means as they stand, which of them would actually drive the distances?* The table measures each
column's share of the total variance across all numeric and 0/1 columns — the quantity a raw Euclidean
distance is built from. `wh_est_year` is measured as the preprocessing step left it, because that is the column a model
would receive.

In [ ]:
numeric_all = performance + conditions_numeric + conditions_binary

scale = pd.DataFrame({
    "role": ["performance"] * len(performance) + ["condition"] * (len(conditions_numeric) + len(conditions_binary)),
    "min": df[numeric_all].min(),
    "max": df[numeric_all].max(),
    "std": df[numeric_all].std().round(3),
    "share_of_total_variance_pct": (100 * df[numeric_all].var() / df[numeric_all].var().sum()).round(5),
})
scale.sort_values("std", ascending=False)

> **Interpretation.**
>
> - Left as they are, these columns could not take part in a distance calculation on equal
>   terms. **`product_wg_ton` alone holds 99.18% of the total variance** across the 17 numeric and 0/1 columns,
>   and `retail_shop_num` a further 0.82%. Every other column — including four of the five performance
>   measures — holds less than 0.003%.
>
> - A raw Euclidean distance would therefore amount to "difference in shipment weight, plus a little difference
>   in retail shop count". K-Means would segment on shipment weight almost alone and ignore breakdowns, storage
>   issues, transport issues and refills entirely. That reflects units, not importance — a ton and a breakdown
>   are not comparable quantities. **Some form of scaling is required before clustering**; which scaler is
>   decided in NB 12.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), performance):
    if df[c].nunique() <= 60:      # whole-number counts: one bar per value
        sns.histplot(data=plot_df, x=c, hue="warehouse", multiple="stack", discrete=True, ax=ax)
    else:
        sns.histplot(data=plot_df, x=c, hue="warehouse", multiple="stack", bins=40, ax=ax)
    ax.set_title(c, fontsize=10); ax.set_xlabel(""); ax.set_ylabel("")
axes.ravel()[-1].axis("off")
fig.suptitle("Performance measures — unrated warehouses stacked on top", fontweight="bold", y=1.001)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - Read from the histograms:
>   - **Storage issues and breakdowns** are where the unrated warehouses stand apart. They make up the entire bar
>     at zero in both, and no rated warehouse sits at zero on either. They also occupy the lowest end of
>     shipment weight.
>   - **Refill requests** tell the opposite story: the unrated warehouses form a thin, roughly even layer across
>     every value from 0 to 8. On refills they look like everyone else, consistent with the preliminary analysis.
>   - **Transport issues** are concentrated at zero in both groups; most unrated warehouses report none.
>   - The shapes match what the preliminary analysis found: refills flat apart from a lower bar at 2; transport issues and breakdowns
>     weighted toward low counts; storage issues jagged, with the gap between 0 and 4.
>
> - For clustering, one point matters most: the unrated warehouses are set apart on **two measures at once** —
>   zero storage issues and zero breakdowns — so any method given both columns will find them easy to
>   separate.

---
## 3. Univariate — the conditions

Under the proposed roles, conditions are what cluster profiles will be *explained* with. Two things
matter here: whether each condition varies enough to explain anything, and — in case the clustering
ends up using conditions directly — how many columns encoding the categorical ones would add.

`wh_est_year` is plotted from **recorded years only** (`wh_est_year_missing == 0`): the preprocessing step filled
49.46% of rows at 2009, which would show as a false spike.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, c in zip(axes.ravel(), conditions_numeric):
    s = df.loc[df["wh_est_year_missing"] == 0, c] if c == "wh_est_year" else df[c]
    if s.nunique() <= 60:
        sns.histplot(s, discrete=True, ax=ax)
    else:
        sns.histplot(s, bins=40, ax=ax)
    ax.set_title(c + ("  (recorded years only)" if c == "wh_est_year" else ""), fontsize=10)
    ax.set_xlabel(""); ax.set_ylabel("")
axes.ravel()[-1].axis("off")
fig.suptitle("Numeric conditions", fontweight="bold", y=1.001)
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - Read from the histograms:
>   - **Flat:** `wh_est_year` (recorded years only), `distributor_num` and `dist_from_hub` — each value roughly as
>     common as any other. The bars for `dist_from_hub` alternate between two heights only because its 217
>     distinct values are divided into 40 bins, so some bins hold five values and some six. That pattern belongs
>     to the plot, not the data.
>   - **One peak with a right tail:** `workers_num` and `retail_shop_num`. The tallest `workers_num` bar, at 30,
>     is inflated by the 733 values that the preprocessing step filled there (another 257 were filled at 24) — a known consequence of
>     that treatment, not a feature of staffing.
>   - `Competitor_in_mkt` is concentrated at 2 to 4 competitors, and `govt_check_l3m` is irregularly spiked, as
>     the preliminary analysis found.
>
> - For the flat columns there is no typical value. A cluster profile that reports, say, the average distance
>   from hub for such a column is reporting the middle of a flat range, not what a typical warehouse looks
>   like. §5 tests which of these conditions relate to performance at all.

In [ ]:
rows = []
for c in conditions_binary + conditions_categorical:
    shares = df[c].value_counts(normalize=True)
    rows.append({
        "condition": c,
        "levels": df[c].nunique(),
        "most_common_level": shares.index[0],
        "most_common_pct": round(100 * shares.iloc[0], 2),
        "least_common_level": shares.index[-1],
        "least_common_pct": round(100 * shares.iloc[-1], 2),
    })

n_dummies = sum(df[c].nunique() for c in conditions_categorical)
n_dummies_dropped = sum(df[c].nunique() - 1 for c in conditions_categorical)
print(f"one-hot encoding the {len(conditions_categorical)} categorical conditions would create "
      f"{n_dummies} columns ({n_dummies_dropped} with one level of each dropped)\n")
pd.DataFrame(rows)

> **Interpretation.**
>
> - Several conditions are dominated by a single level: `is_unrated_warehouse` is 0 for
>   96.37% of warehouses, `flood_proof` 0 for 94.54%, `Location_type` Rural for 91.83% and `flood_impacted`
>   0 for 90.18%. A condition shared by nine warehouses in ten can only explain differences among the few that
>   do not share it. The rarest single level is `zone == East` at 1.72%.
>
> - If the six categorical conditions were given directly to a clustering method, one-hot encoding would add
>   **23 columns** (17 with one level of each dropped) — more than all 17 numeric and 0/1 columns put together.
>   Every one of those dummy columns counts in the distance calculation, so the categorical conditions would
>   outweigh everything else by sheer number, whether or not they relate to performance. §5 shows whether they
>   do.


---
## 4. Bivariate — do the performance measures duplicate each other?

A distance adds up every column it is given. If two columns carry the same information, that
information is **counted twice**, and the clusters lean toward it for no reason other than
duplication. The preliminary analysis already reported some of these relationships across all numeric columns; here
all ten pairs of performance measures are examined together, in two ways:

- **Pearson and Spearman** side by side — a straight-line and a rank-based measure.
- **With and without the 908 unrated warehouses.** Those warehouses have zero storage issues *and* zero
  breakdowns (as found in the preliminary analysis). A block of rows sitting at zero on two columns at once can create or inflate
  a correlation on its own, so it is worth knowing how much of each relationship survives without them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax, method in zip(axes, ["pearson", "spearman"]):
    sns.heatmap(df[performance].corr(method=method), annot=True, fmt=".3f", cmap="coolwarm",
                center=0, vmin=-1, vmax=1, cbar=False, ax=ax)
    ax.set_title(f"{method.title()} correlation — performance measures")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

rated = df["is_unrated_warehouse"] == 0
rows = []
for i, a in enumerate(performance):
    for b in performance[i + 1:]:
        rows.append({
            "measure_a": a, "measure_b": b,
            "pearson_all_25000": round(df[a].corr(df[b]), 3),
            "pearson_without_unrated": round(df.loc[rated, a].corr(df.loc[rated, b]), 3),
        })
pairs = pd.DataFrame(rows)
pairs["change"] = (pairs["pearson_without_unrated"] - pairs["pearson_all_25000"]).round(3)
pairs.sort_values("pearson_all_25000", key=abs, ascending=False).reset_index(drop=True)

> **Interpretation.**
>
> - One pair is a near-duplicate; the rest of the relationships are weak or absent.
>
> - **`product_wg_ton` ↔ `storage_issue_reported_l3m`: Pearson 0.987, Spearman 0.989.** Removing the 908
>   unrated warehouses does not weaken it — it rises slightly, to 0.991. The relationship belongs to the rated
>   network itself, not to the block of zeros.
> - **Storage issues ↔ breakdowns (0.377) and shipment weight ↔ breakdowns (0.343)** are moderate, and both
>   fall once the unrated warehouses are removed, to 0.271 and 0.263. Roughly a quarter of each relationship
>   is the 908 warehouses sitting at zero on both columns at once.
> - **Transport issues** relate weakly and negatively to shipment weight (−0.174; −0.189 without the unrated
>   warehouses) and to storage issues (−0.144; −0.167).
> - **Refill requests are unrelated to every other measure** — no pair exceeds 0.021 in absolute value, with or
>   without the unrated warehouses.
> - Pearson and Spearman agree throughout, so no relationship is hidden by a curved shape.
>
> - **For clustering:** giving K-Means both shipment weight and storage issues would count one underlying
>   dimension twice, and the segments would lean toward it purely because of the duplication. Refill requests
>   and transport issues, by contrast, carry information no other measure repeats.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))
sns.scatterplot(data=plot_df, x="storage_issue_reported_l3m", y="product_wg_ton", hue="warehouse",
                s=6, alpha=0.3, linewidth=0, ax=axes[0])
axes[0].set_title("Storage issues against shipment weight — every warehouse")
sns.boxplot(data=df, x="wh_breakdown_l3m", y="storage_issue_reported_l3m", ax=axes[1])
axes[1].set_title("Storage issues at each breakdown count")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - Read from the plots:
>   - **Left.** Shipment weight rises in a narrow diagonal band as storage issues rise. Each storage-issue count
>     occupies a short vertical strip of shipment weights that overlaps its neighbours, and the band runs
>     continuously from 4 storage issues to 39 with no gap along it. The unrated warehouses form a separate
>     vertical strip at zero storage issues, with shipments up to roughly 14,000 t.
>   - **Right.** Storage issues rise with breakdowns only up to a point: the median climbs from about 7 at one
>     breakdown to about 17–18 at two or three and about 20 at four — then stays at about 20 through five and
>     six. Beyond four breakdowns, more breakdowns come with no more storage issues. Zero breakdowns means zero
>     storage issues: the unrated group again.


---
## 5. Bivariate — which conditions are related to performance?

Cluster profiling explains segments using conditions. If no condition is related to performance, the
profiles will have nothing to explain the segments with — so this is checked before any model is built.

**Categorical and 0/1 conditions against each performance measure.** This section applies analysis of
variance. Two tests are run on each pair:

- **One-way ANOVA** — do the group means differ?
- **Kruskal–Wallis** — the same question without assuming normally distributed values, which matters
  because four of the five measures are counts with the irregular shapes seen in §2.

With 25,000 rows both tests will call almost any difference significant, so neither p-value can say
whether a difference *matters*. The deciding figure is **eta squared (η²)**: the share of a measure's
variance explained by which group a warehouse belongs to. A common rule of thumb reads η² of 0.01 as
small, 0.06 as medium and 0.14 as large. The heatmap shows η² as a percentage.


In [ ]:
rows = []
for cond in conditions_binary + conditions_categorical:
    for perf in performance:
        samples = [g.to_numpy() for _, g in df.groupby(cond)[perf]]
        _values = df[perf]; _groups = df[cond]
        grand_mean = _values.mean()
        between_ss = sum(grp.size() * (grp.mean() - grand_mean)**2 for _, grp in _values.groupby(_groups))
        total_ss = ((_values - grand_mean)**2).sum()
        eta2 = between_ss / total_ss
        rows.append({
            "condition": cond,
            "measure": perf,
            "eta_sq": eta2,
            "anova_p": stats.f_oneway(*samples).pvalue,
            "kruskal_p": stats.kruskal(*samples).pvalue,
        })
assoc = pd.DataFrame(rows)

eta_pct = 100 * assoc.pivot(index="condition", columns="measure", values="eta_sq")[performance]
plt.figure(figsize=(11, 6))
sns.heatmap(eta_pct, annot=True, fmt=".2f", cmap="rocket_r", vmin=0,
            cbar_kws={"label": "η² — % of the measure's variance explained"})
plt.title("How much of each performance measure do the categorical and 0/1 conditions explain?",
          fontweight="bold")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

significant = assoc["kruskal_p"] < 0.05
print(f"condition-measure pairs tested          : {len(assoc)}")
print(f"significant at 5% (Kruskal-Wallis)      : {int(significant.sum())}")
print(f"significant AND eta squared below 0.01  : {int((significant & (assoc['eta_sq'] < 0.01)).sum())}")
print(f"eta squared of 0.01 or more             : {int((assoc['eta_sq'] >= 0.01).sum())}")
print("\npairs with eta squared of 0.01 or more:")
print(assoc[assoc["eta_sq"] >= 0.01].sort_values("eta_sq", ascending=False)
      .assign(eta_sq_pct=lambda t: (100 * t["eta_sq"]).round(2),
              anova_p=lambda t: t["anova_p"].map("{:.3g}".format),
              kruskal_p=lambda t: t["kruskal_p"].map("{:.3g}".format))
      .drop(columns="eta_sq").to_string(index=False))

> **Interpretation.**
>
> - Of the 55 condition–measure pairs, **19 are statistically significant, and 11 of those 19
>   explain less than 1%** of the measure's variance. At this sample size, significance mostly reflects the
>   number of rows — which is why η² is the figure that decides.
>
> - Only **8 pairs reach 1%**, and they involve just three conditions:
>   - **Certificate** — 16.64% of breakdowns, 15.02% of storage issues, 11.09% of shipment weight.
>   - **`is_unrated_warehouse`** — 15.99%, 13.18% and 7.78% of the same three. The certificate's levels include
>     `Unrated`, so much of the certificate's effect may simply be the unrated group; the next cell checks.
>   - **Temperature regulation** — 6.81% of refill requests, a medium effect and the only sizeable association
>     with refills anywhere, plus 1.03% of shipment weight.
>
> - Every other categorical or 0/1 condition explains under 1% of every measure. `Location_type` comes closest
>   (0.56% of shipment weight, 0.62% of storage issues). Capacity size, zone, regional zone, ownership, electric
>   back-up and both flood indicators explain **0.03% or less** of every measure — consistent with the finding that **location and ownership are weak**.


**The same question without the 908 unrated warehouses.** The conditions most strongly related to
performance above involve the certificate. But the certificate includes the `Unrated` level, and those
908 warehouses sit at zero on storage issues and breakdowns (as established in the preliminary analysis) — a block that could produce most
of the association on its own. Re-running the η² calculation on the 24,092 rated warehouses shows how
much any condition explains once that block is set aside. `is_unrated_warehouse` is left out here because
it takes a single value among rated warehouses.

In [ ]:
rated = df["is_unrated_warehouse"] == 0
rows = []
for cond in [c for c in conditions_binary + conditions_categorical if c != "is_unrated_warehouse"]:
    for perf in performance:
        # eta squared — all warehouses
        _values_all = df[perf]; _groups_all = df[cond]
        _gm_all = _values_all.mean()
        _bss_all = sum(grp.size() * (grp.mean() - _gm_all)**2 for _, grp in _values_all.groupby(_groups_all))
        _tss_all = ((_values_all - _gm_all)**2).sum()
        _eta_all = _bss_all / _tss_all

        # eta squared — rated warehouses only
        _values_r = df.loc[rated, perf]; _groups_r = df.loc[rated, cond]
        _gm_r = _values_r.mean()
        _bss_r = sum(grp.size() * (grp.mean() - _gm_r)**2 for _, grp in _values_r.groupby(_groups_r))
        _tss_r = ((_values_r - _gm_r)**2).sum()
        _eta_r = _bss_r / _tss_r

        rows.append({"condition": cond, "measure": perf,
                     "eta_sq_pct_all": round(100 * _eta_all, 2),
                     "eta_sq_pct_rated_only": round(100 * _eta_r, 2)})
rated_assoc = pd.DataFrame(rows)

print(f"pairs with eta squared of 1% or more — all warehouses: {int((rated_assoc['eta_sq_pct_all'] >= 1).sum())}"
      f"   rated only: {int((rated_assoc['eta_sq_pct_rated_only'] >= 1).sum())}\n")
print("pairs reaching 0.5% in either column:")
print(rated_assoc[(rated_assoc["eta_sq_pct_all"] >= 0.5) | (rated_assoc["eta_sq_pct_rated_only"] >= 0.5)]
      .sort_values("eta_sq_pct_all", ascending=False).to_string(index=False))

> **Interpretation.**
>
> - Setting the 908 aside changes the picture sharply.
>
> | Condition → measure | All warehouses | Rated only |
> |---|---|---|
> | certificate → breakdowns | 16.64% | **0.77%** |
> | certificate → storage issues | 15.02% | **2.12%** |
> | certificate → shipment weight | 11.09% | **3.60%** |
> | temperature regulation → refill requests | 6.81% | **7.12%** |
> | temperature regulation → shipment weight | 1.03% | 0.50% |
> | location type → storage issues | 0.62% | 0.38% |
> | location type → shipment weight | 0.56% | 0.37% |
>
> - **Almost all of the certificate's link with breakdowns was the unrated group.** Among rated warehouses the five
>   grades still explain a small share of shipment weight (3.60%) and storage issues (2.12%), but almost none
>   of breakdowns (0.77%). The one association *not* produced by the unrated group is **temperature regulation
>   with refill requests**, which holds — slightly stronger, at 7.12%.
>
> - Among the 24,092 rated warehouses, only **3 of 50** condition–measure pairs reach 1%. Taken together, the
>   categorical and on/off conditions explain very little of how rated warehouses perform.


**Numeric conditions against each performance measure.** Spearman correlation, because several of
these columns are counts or near-uniform (§3) and a rank-based measure makes no assumption about
straight-line shape. `wh_est_year` is correlated on **recorded years only** — the preprocessing step showed the filled
column understates its relationships (with storage issues, −0.859 on recorded years against −0.629
after filling).

In [ ]:
recorded_year = df["wh_est_year_missing"] == 0
spearman = {}
for cond in conditions_numeric:
    rows_used = recorded_year if cond == "wh_est_year" else pd.Series(True, index=df.index)
    spearman[cond] = {perf: df.loc[rows_used, cond].corr(df.loc[rows_used, perf], method="spearman")
                      for perf in performance}
spearman = pd.DataFrame(spearman).T[performance]

plt.figure(figsize=(11, 5))
sns.heatmap(spearman, annot=True, fmt=".3f", cmap="coolwarm", center=0, vmin=-1, vmax=1,
            cbar_kws={"label": "Spearman correlation"})
plt.title("Numeric conditions against performance measures\n(wh_est_year: recorded years only)",
          fontweight="bold")
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

strongest = spearman.abs().max(axis=1).sort_values(ascending=False)
print("strongest absolute correlation of each condition with any performance measure:")
print(strongest.round(3).to_string())

> **Interpretation.**
>
> - Of the seven numeric conditions, **only establishment year relates to performance.** On
>   recorded years, warehouses established earlier report more storage issues (ρ −0.872), ship more (−0.850)
>   and, more weakly, report more breakdowns (−0.380). Establishment year is unrelated to refill requests
>   (0.016) and transport issues (−0.014).
>
> - Every other numeric condition — workers, distance from hub, competitors, retail shops, distributors and
>   government checks — has no correlation stronger than 0.015 in absolute value with any performance measure.
>   For practical purposes they are unrelated to performance.
>
> - Across §5, then, the conditions with any real relationship to performance are few: **warehouse age,
>   certificate grade (largely through the unrated group), and temperature regulation (with refills).** Those are
>   the conditions cluster profiles can expect to differ on; the rest are unlikely to differ meaningfully between
>   segments.


**Are the conditions related to each other?** This section also applies chi-square tests of
association between features. Among the categorical and 0/1 conditions, strong association would mean
two conditions tell the same story — which matters when profiles are read, because a segment that
differs on one will appear to differ on the other too. Chi-square answers *whether* two conditions are
associated; **Cramér's V** answers *how strongly*, on a scale from 0 (none) to 1 (one determines the
other).


In [ ]:
group_conditions = conditions_binary + conditions_categorical
V = pd.DataFrame(index=group_conditions, columns=group_conditions, dtype=float)
for a in group_conditions:
    for b in group_conditions:
        if a == b:
            V.loc[a, b] = 1.0
        else:
            _chi2 = stats.chi2_contingency(pd.crosstab(df[a], df[b]), correction=False)[0]
            _n = len(df[a])
            _r, _k = pd.crosstab(df[a], df[b]).shape
            V.loc[a, b] = np.sqrt(_chi2 / (_n * (min(_r, _k) - 1)))

plt.figure(figsize=(10, 8))
sns.heatmap(V, annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1, annot_kws={"size": 8},
            cbar_kws={"label": "Cramér's V"})
plt.title("Association between categorical and 0/1 conditions", fontweight="bold")
plt.tight_layout(); plt.show()

rows = []
for i, a in enumerate(group_conditions):
    for b in group_conditions[i + 1:]:
        p = stats.chi2_contingency(pd.crosstab(df[a], df[b]), correction=False)[1]
        rows.append({"condition_a": a, "condition_b": b, "cramers_v": round(V.loc[a, b], 3),
                     "chi2_p": f"{p:.3g}"})
print("strongest associations:")
print(pd.DataFrame(rows).sort_values("cramers_v", ascending=False).head(8).to_string(index=False))

> **Interpretation.**
>
> - Most conditions are close to independent of one another, with three clear exceptions.
>
> - **`is_unrated_warehouse` ↔ certificate, V = 1.00** — by construction, since the flag is defined from the
>   certificate. Not a finding.
> - **`WH_capacity_size` ↔ `WH_regional_zone`, V = 0.847.** A warehouse's regional zone very nearly determines
>   its capacity size. The two largely describe the same thing, so a segment that differs on one will appear to
>   differ on the other; they should be read together, not as two independent explanations.
> - **Temperature regulation ↔ certificate, V = 0.458** — a moderate association, far stronger than temperature
>   regulation's association with the unrated flag alone (0.123). It therefore reflects the grades themselves,
>   not only the unrated group.
>
> - Electric back-up ↔ ownership (0.230) is weak to moderate; every other pair is 0.18 or below.
>
> - **Zone and regional zone are only weakly associated (V = 0.178)**, although the data dictionary describes
>   regional zone as sitting "under each zone". If regional zones were nested inside zones, V would be close to
>   1. In this data they are not nested.
>
> - Every pair listed is significant, with p far below 0.05. At this sample size the chi-square test on its own
>   says little; Cramér's V is the figure to read.

---
## 6. Multivariate — is there structure for clustering to find?

Clustering will always return clusters, whether or not the data contains any groups. So before choosing
a number of clusters, it is worth asking whether the data shows any grouping at all, and in which
set of columns. Three views:

- **PCA of the performance measures** — how many independent directions of variation the five measures
  really contain, and which measures define each. Measures are standardised first (mean 0, standard
  deviation 1) so that §2's scale differences do not decide the result; this is exploratory only, and the
  scaling decision itself belongs to the transformation notebook.
- **A 2-D map** of the warehouses on the first two components.
- **The Hopkins statistic** — a direct test of clustering tendency. It compares how close real warehouses
  are to their nearest neighbour with how close randomly placed points are to their nearest warehouse.
  A value near **0.5** means the data is no more grouped than random points; values approaching **1**
  mean clear grouping.

In [ ]:
X_perf = StandardScaler().fit_transform(df[performance])
pca = PCA(random_state=RANDOM_STATE).fit(X_perf)

components = [f"PC{i + 1}" for i in range(len(performance))]
explained = pd.DataFrame({
    "component": components,
    "variance_explained_pct": (100 * pca.explained_variance_ratio_).round(2),
    "cumulative_pct": (100 * pca.explained_variance_ratio_.cumsum()).round(2),
})
print(explained.to_string(index=False))
print("\nloadings — how strongly each measure defines each component:")
pd.DataFrame(pca.components_.T, index=performance, columns=components).round(3)

> **Interpretation.**
>
> - The five standardised measures contain **four real directions of variation, not five.**
>
> | Component | Variance | Defined by |
> |---|---|---|
> | PC1 | 44.71% | shipment weight (0.644) and storage issues (0.647) together, with breakdowns (0.374) — a "volume and problems" direction |
> | PC2 | 20.46% | transport issues (0.732) and refill requests (0.592) |
> | PC3 | 19.75% | refill requests (0.806) set against transport issues (−0.528) |
> | PC4 | 14.83% | mainly breakdowns (0.821) |
> | PC5 | **0.24%** | shipment weight against storage issues (−0.703 vs 0.710) — the part in which the two differ |
>
> - PC5 is §4's near-duplicate seen from another angle: the only thing separating shipment weight from storage
>   issues carries a quarter of one percent of the variance. The first four components account for 99.76%.
>   PCs 2 to 4 are of similar size, which fits §4's finding that refills, transport issues and breakdowns each
>   vary largely on their own.


In [ ]:
scores = pca.transform(X_perf)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.scatterplot(x=scores[:, 0], y=scores[:, 1], hue=plot_df["warehouse"], s=5, alpha=0.3,
                linewidth=0, ax=axes[0])
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_title("Warehouses on the first two components — all")
sns.scatterplot(x=scores[rated.to_numpy(), 0], y=scores[rated.to_numpy(), 1], s=5, alpha=0.3,
                linewidth=0, color="#4C72B0", ax=axes[1])
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("Same map — rated warehouses only")
plt.tight_layout(); plt.show()

> **Interpretation.**
>
> - Read from the plots: the rated warehouses form **one continuous cloud**, with no empty space
>     dividing it into groups. The horizontal striping comes from the count columns' few distinct values, not from
>     groups. The unrated warehouses sit at the far left of the first component — low volume, no storage issues or
>     breakdowns — adjoining the edge of the main cloud rather than separated from it by a clear gap. With them
>     removed (right), the cloud keeps the same shape.
>
> - If clustering finds segments in these measures, they will be divisions of a continuous spread, not naturally
>   separate groups.


**Hopkins statistic.** Computed on four candidate sets of columns, each standardised, using 2,000 sampled
warehouses (8% of rows) and repeated with five different random samples to show how stable the value is.

**A high value has two possible causes, and the test alone cannot tell them apart.**

1. *Genuine groups* — warehouses gathered into separate clouds with empty space between them.
2. *Columns with few distinct values.* Four of the five performance measures are counts. That places every
   warehouse on a grid: neighbours often differ in one column only, while a randomly placed point usually
   falls *between* grid lines, far from any warehouse. The statistic rises toward 1 with no groups at all.
   In the extreme, two warehouses sit at exactly the same point and the nearest-neighbour distance is zero —
   `pct_exact_duplicates` reports how often that happens.

To separate the two causes, every set is scored twice: once as recorded, and once after **shuffling each
column independently**. Shuffling keeps every column's values — and so the grid — exactly as they are, but
breaks any tendency for particular values to occur *together*, which is what a real group is. If the
shuffled data scores as high as the real data, the high value comes from the grid, not from groups. The
column `structure_beyond_shuffled` is the difference.

The sets are chosen to isolate where any structure comes from: all five measures; the five without each
member of the near-duplicate pair found in §4 in turn; the five on rated warehouses only; and every numeric
and 0/1 column together.


In [ ]:
candidate_sets = {
    "performance measures (5)": (performance, pd.Series(True, index=df.index)),
    "performance without product_wg_ton (4)": ([c for c in performance if c != "product_wg_ton"],
                                               pd.Series(True, index=df.index)),
    "performance without storage_issue_reported_l3m (4)": ([c for c in performance if c != "storage_issue_reported_l3m"],
                                                           pd.Series(True, index=df.index)),
    "performance measures, rated warehouses only (5)": (performance, rated),
    "performance + numeric and 0/1 conditions (17)": (numeric_all, pd.Series(True, index=df.index)),
}

rows = []
for name, (cols, rows_used) in candidate_sets.items():
    X = StandardScaler().fit_transform(df.loc[rows_used, cols])

    # Same values in every column, but each column shuffled on its own: the grid survives,
    # any tendency for values to occur together does not.
    rng = np.random.default_rng(RANDOM_STATE)
    X_shuffled = np.column_stack([rng.permutation(X[:, j]) for j in range(X.shape[1])])

    # Compute Hopkins statistic for real data across 5 seeds.
    # Near 0.5: no more grouped than random points. Toward 1: clearly grouped.
    real = []
    for i in range(5):
        _rng = np.random.default_rng(RANDOM_STATE + i)
        _n, _d = X.shape
        _real_pts = X[_rng.choice(_n, 2000, replace=False)]
        _uniform = _rng.uniform(X.min(axis=0), X.max(axis=0), size=(2000, _d))
        _nn = NearestNeighbors(n_neighbors=2).fit(X)
        _u = _nn.kneighbors(_uniform, n_neighbors=1)[0][:, 0]   # random point -> nearest warehouse
        _w = _nn.kneighbors(_real_pts, n_neighbors=2)[0][:, 1]  # warehouse -> nearest other warehouse
        real.append((_u.sum() / (_u.sum() + _w.sum()), 100 * (_w == 0).mean()))

    # Compute Hopkins statistic for shuffled data across 5 seeds.
    shuffled = []
    for i in range(5):
        _rng = np.random.default_rng(RANDOM_STATE + i)
        _n, _d = X_shuffled.shape
        _real_pts = X_shuffled[_rng.choice(_n, 2000, replace=False)]
        _uniform = _rng.uniform(X_shuffled.min(axis=0), X_shuffled.max(axis=0), size=(2000, _d))
        _nn = NearestNeighbors(n_neighbors=2).fit(X_shuffled)
        _u = _nn.kneighbors(_uniform, n_neighbors=1)[0][:, 0]
        _w = _nn.kneighbors(_real_pts, n_neighbors=2)[0][:, 1]
        shuffled.append((_u.sum() / (_u.sum() + _w.sum()), 100 * (_w == 0).mean()))

    h_real = np.array([r[0] for r in real])
    h_shuffled = np.array([r[0] for r in shuffled])

    rows.append({
        "column_set": name,
        "rows": int(rows_used.sum()),
        "hopkins_real": round(h_real.mean(), 3),
        "hopkins_shuffled": round(h_shuffled.mean(), 3),
        "structure_beyond_shuffled": round(h_real.mean() - h_shuffled.mean(), 3),
        "sd_across_samples": round(h_real.std(), 3),
        "pct_exact_duplicates": round(np.mean([r[1] for r in real]), 2),
    })
pd.DataFrame(rows)

> **Interpretation.**
>
> - Taken at face value every set looks strongly clustered — Hopkins between 0.789 and 0.971,
>   stable to within 0.002 across samples. **The shuffled scores show that most of it is the grid, not groups.**
>
> | Column set | Real | Shuffled | Beyond shuffled |
> |---|---|---|---|
> | performance measures (5) | 0.963 | 0.742 | **0.221** |
> | without shipment weight (4) | 0.971 | 0.945 | 0.026 |
> | without storage issues (4) | 0.967 | 0.938 | 0.030 |
> | rated warehouses only (5) | 0.960 | 0.744 | 0.216 |
> | all numeric and 0/1 columns (17) | 0.789 | 0.749 | 0.040 |
>
> - The five measures clear their shuffled version by 0.221 — but **that margin disappears when either
>   shipment weight or storage issues is removed**, falling to 0.026 or 0.030. The only strong joint structure
>   in the performance data is the near-straight line between those two columns (§4). A thin line is far from
>   random scatter, and Hopkins registers it, but a line is not a set of separate groups.
> - Removing the unrated warehouses barely changes the result (0.221 → 0.216): at 3.6% of the sample they do
>   not drive the statistic.
> - Without shipment weight, **91.83%** of sampled warehouses have an exact duplicate — the four count columns
>   alone place most warehouses on shared points. Without storage issues the figure is 6.18%, and with all five
>   measures 2.54%.
> - Adding every numeric and 0/1 condition leaves a margin of just 0.040.
>
> - **No candidate set shows meaningful natural grouping. The performance data is a continuum.** Clustering can
>   still divide it into segments that are useful for describing the network — tiers of warehouses — but those
>   segments will be cuts through a continuous spread, so separation measures such as the silhouette score and
>   the Dunn index should be expected to be **modest**. The expectation is recorded here, before any model is
>   fitted, so that a modest score in NB 15 is read correctly rather than mistaken for a failed model.


In [ ]:
sample = plot_df.sample(3000, random_state=RANDOM_STATE)
g = sns.pairplot(sample[performance + ["warehouse"]], hue="warehouse", corner=True, diag_kind="hist",
                 plot_kws={"s": 8, "alpha": 0.4, "linewidth": 0}, height=2.3)
g.figure.suptitle("Performance measures, pairwise — 3,000 sampled warehouses", fontweight="bold", y=1.01)
plt.show()

> **Interpretation.**
>
> - Read from the pairplot of 3,000 sampled warehouses: every pair of measures fills a grid of
>   lines or points rather than forming separate clouds. The only tight shape is shipment weight against storage
>   issues — the diagonal band. Refill requests show no pattern against anything. Two edges are visible:
>   warehouses with a single breakdown have lower shipments and fewer storage issues than the rest, and the
>   highest shipment weights occur only among warehouses with few transport issues. The unrated warehouses sit
>   in one corner of every panel involving storage issues or breakdowns.
>
> - Nothing here suggests natural groups beyond that corner — consistent with the PCA map and the Hopkins
>   scores.


---
## 7. What the evidence means for the clustering design

| Finding | Evidence | Implication |
|---|---|---|
| Shipment weight dominates raw distances | §2 — 99.18% of total variance | scaling is required |
| Shipment weight and storage issues duplicate each other | §4 — r = 0.987 (0.991 without unrated); §6 — PC5 holds 0.24% | only one of the pair should define segments |
| Without shipment weight, the measures collapse onto shared points | §6 — 91.83% exact duplicates without it; 6.18% without storage issues | keep shipment weight as the input; use storage issues for profiling |
| Refills, transport issues and breakdowns vary largely independently | §4; §6 — PC2 to PC4 of similar size | each carries information of its own |
| Conditions barely relate to performance | §5 — among rated warehouses 3 of 50 pairs reach η² 1%; among numeric conditions only establishment year | conditions should profile segments, not define them |
| Categorical conditions would outnumber everything else | §3 — 23 one-hot columns | they would dominate distances by number alone |
| All columns together show little structure | §6 — Hopkins margin 0.040 over shuffled | no case for clustering on everything |
| The performance data is a continuum | §6 — margin vanishes without either duplicate; PCA map is one cloud | segments will be tiers; separation scores will be modest |
| The 908 unrated warehouses are identified by a rule and sit at the cloud's edge | §2, §4, §6; preliminary analysis | a cluster spent on them would only rediscover that rule |

Taken together, the evidence points to four design choices for the clustering. Segments should be defined by performance measures alone, with conditions used afterwards to profile and explain them — adding conditions to the clustering would inject many columns that barely relate to performance and would dominate the distance count by number alone. Of the near-duplicate pair, shipment weight is the better clustering input: without it, 91.83% of warehouses share an exact point with another warehouse, whereas without storage issues only 6.18% do, and shipment weight is also a direct outcome measure. Scaling is required before any distance is computed, since unscaled shipment weight holds 99.18% of the variance; which scaler is most appropriate is examined in the transformation notebook. The 908 unrated warehouses are already identified by their recorded attributes and sit at the edge of the performance cloud — assigning them a segment by rule avoids clustering rediscovering a known group, while still giving every one of the 25,000 warehouses a segment. Because the performance data is a continuum rather than a set of naturally separate groups, separation measures such as the silhouette score should be expected to be modest when the model is evaluated — recording that expectation here, before any model is fitted, ensures a modest score is read correctly rather than mistaken for a failed model.

---
## 8. Checks

This notebook is exploratory. It must not have altered the data it read.


In [ ]:
assert df.equals(load_preprocessed()), "the dataframe was modified — this notebook must only report"
assert df.shape == (25_000, 25)
print(f"data unchanged: {df.shape[0]:,} rows x {df.shape[1]} columns; nothing written by this notebook")

---
## Summary

**What Objective 1's data looks like from a clustering point of view.**

1. **Scale.** Shipment weight holds 99.18% of raw variance, so unscaled distances would be shipment weight alone.
2. **Redundancy.** Shipment weight and storage issues are near-duplicates (r = 0.987; the component separating
   them holds 0.24% of variance). Refill requests, transport issues and breakdowns each vary largely on their
   own; refills are unrelated to every other measure.
3. **Conditions.** Few relate to performance. Among rated warehouses, only temperature regulation with refills
   (η² 7.12%), certificate grade with shipment weight (3.60%) and storage issues (2.12%), and establishment year
   (ρ −0.872 with storage issues, −0.850 with shipment weight, −0.380 with breakdowns) show a real relationship.
   Location, zone, capacity, ownership, electric back-up, flood exposure, workers, distance, competitors, retail
   shops, distributors and government checks are practically unrelated.
4. **Among the conditions,** regional zone very nearly determines capacity size (V = 0.847), and zone and
   regional zone are not nested (V = 0.178).
5. **Structure.** There is no natural grouping. High Hopkins scores come from the grid of count values and the
   shipment–storage line; the margin over shuffled data vanishes once either member of that pair is removed.
6. **The 908 unrated warehouses** are set apart at zero storage issues and zero breakdowns, sit at the edge of
   the continuous cloud, and do not drive the structure statistics.

**Connection to the project expectation.** Its expectation that *"shipment weight is driven mainly by reported storage issues
and warehouse age, while location and ownership appear weak"* is supported at the EDA stage — as association,
not cause, since the data is a single snapshot.

**Handed to the transformation notebook:** the four design choices described in §7 — the column set, how to handle the near-duplicate pair, the need for scaling, and the rule-based treatment of the unrated warehouses — along with the expectation that separation scores will be modest when the model is evaluated.